# Generative AI Assignment 1


## Setup and Imports

In [ ]:
import subprocess
import sys

packages = ['langchain', 'langchain-groq', 'python-dotenv', 'pandas', 'numpy']

for package in packages:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
    except:
        pass

print("Packages installed successfully!")

Packages installed successfully!


In [ ]:
import os
import json
import pandas as pd
import numpy as np
from typing import Dict, List, Optional
import warnings
warnings.filterwarnings('ignore')

# LangChain imports
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain_groq import ChatGroq


print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:

import getpass
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ API Key: ")

# Initialize the LLM
llm = ChatGroq(
    temperature=0.3,  
    model_name="openai/gpt-oss-120b",  
)

print("LLM initialized successfully!")
print(f"Model: {llm.model_name}")
print(f"Temperature: {llm.temperature}")

LLM initialized successfully!
Model: openai/gpt-oss-120b
Temperature: 0.3


# PART 1: Topic Detection 


## Step 1: Load and Explore the BBC News Dataset

In [ ]:
bbc_df = pd.read_csv('bbc-news-data.csv', sep='\t')

print("Dataset Shape:", bbc_df.shape)
print("\nColumn Names:", bbc_df.columns.tolist())
print("\nFirst Few Rows:")
print(bbc_df.head())

print("\nData Types:")
print(bbc_df.dtypes)

print("\nMissing Values:")
print(bbc_df.isnull().sum())

print("\nCategory Distribution:")
print(bbc_df['category'].value_counts())

Dataset Shape: (2225, 4)

Column Names: ['category', 'filename', 'title', 'content']

First Few Rows:
   category filename                              title  \
0  business  001.txt  Ad sales boost Time Warner profit   
1  business  002.txt   Dollar gains on Greenspan speech   
2  business  003.txt  Yukos unit buyer faces loan claim   
3  business  004.txt  High fuel prices hit BA's profits   
4  business  005.txt  Pernod takeover talk lifts Domecq   

                                             content  
0   Quarterly profits at US media giant TimeWarne...  
1   The dollar has hit its highest level against ...  
2   The owners of embattled Russian oil giant Yuk...  
3   British Airways has blamed high fuel prices f...  
4   Shares in UK drinks and food firm Allied Dome...  

Data Types:
category    object
filename    object
title       object
content     object
dtype: object

Missing Values:
category    0
filename    0
title       0
content     0
dtype: int64

Category Distribution:


In [ ]:
bbc_df = bbc_df.head(30).reset_index(drop=True)
print(f"Limited dataset to {len(bbc_df)} articles")
print(f"\nCategory Distribution in limited dataset:")
print(bbc_df['category'].value_counts())

Limited dataset to 30 articles

Category Distribution in limited dataset:
category
business    30
Name: count, dtype: int64


## Step 2: Topic Classification 

In [ ]:
topic_classification_template = """You are an expert news categorizer. Analyze the following news article and classify it into ONE of these categories: Business, Entertainment, Politics, Sport, or Tech.

Few-shot examples:
1. Article about stock market crash -> Business
2. Article about celebrity gossip -> Entertainment  
3. Article about government policy -> Politics
4. Article about football match -> Sport
5. Article about new software release -> Tech

Article Title: {title}
Article Content: {content}

Respond with ONLY the category name (Business, Entertainment, Politics, Sport, or Tech). Do not include any explanation."""

topic_prompt = PromptTemplate(
    input_variables=["title", "content"],
    template=topic_classification_template
)

topic_chain = LLMChain(llm=llm, prompt=topic_prompt)
print("Topic Classification Chain created successfully!")

Topic Classification Chain created successfully!


C:\Users\yashr\AppData\Local\Temp\ipykernel_5144\3861908278.py:21: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  topic_chain = LLMChain(llm=llm, prompt=topic_prompt)


In [ ]:
from dotenv import load_dotenv
load_dotenv() 

True

In [9]:
# Test topic classification with first article
test_article = bbc_df.iloc[0]
print("Testing Topic Classification:")
print(f"\nOriginal Category: {test_article['category']}")
print(f"\nTitle: {test_article['title']}")
print(f"\nContent (first 300 chars): {test_article['content'][:300]}...")

detected_topic = topic_chain.run(
    title=test_article['title'],
    content=test_article['content']
)

print(f"\nDetected Topic: {detected_topic.strip()}")

Testing Topic Classification:

Original Category: business

Title: Ad sales boost Time Warner profit

Content (first 300 chars):  Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.  The firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher advert sales. TimeWarner said fo...


C:\Users\yashr\AppData\Local\Temp\ipykernel_5144\2258587902.py:8: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  detected_topic = topic_chain.run(



Detected Topic: Business


In [ ]:
summarization_template = """You are an expert news summarizer. Summarize the following news article in 2-3 sentences, capturing the main points and key information (who/what/when/where/why as applicable). Be concise and factual without any personal commentary.

Article Title: {title}
Article Content: {content}

Summary:"""

summary_prompt = PromptTemplate(
    input_variables=["title", "content"],
    template=summarization_template
)

summary_chain = LLMChain(llm=llm, prompt=summary_prompt)
print("Summarization Chain created successfully!")

Summarization Chain created successfully!


In [ ]:
print("Testing Summarization:")
print(f"\nTitle: {test_article['title']}")
print(f"\nOriginal Content Length: {len(test_article['content'])} characters")

summary = summary_chain.run(
    title=test_article['title'],
    content=test_article['content']
)

print(f"\nGenerated Summary:")
print(summary.strip())

Testing Summarization:

Title: Ad sales boost Time Warner profit

Original Content Length: 2525 characters

Generated Summary:
Time Warner’s quarterly profit jumped 76% to $1.13 billion for the three months ended December, helped by higher high‑speed internet sales, stronger advertising revenue and a one‑off gain that offset a dip at Warner Bros.  The company, now an 8% shareholder in Google, reported mixed results for AOL—losing 464,000 subscribers but seeing an 8% rise in underlying profit before exceptional items—and a 27% fall in film‑division earnings after box‑office flops.  For the full year, profit rose 27% to $3.36 billion, but Time Warner will restate its 2000 and 2003 results and has offered $300 million to settle SEC charges related to its AOL business.


In [ ]:
entity_extraction_template = """You are an expert named entity recognizer. From the following news article, extract all important entities (people names, organizations, locations/places). List them as a comma-separated string.

Guidelines:
- Include proper nouns only
- Include company names, country names, city names
- Include person names mentioned
- Be concise and accurate

Article Title: {title}
Article Content: {content}

Key Entities (comma-separated):"""

entity_prompt = PromptTemplate(
    input_variables=["title", "content"],
    template=entity_extraction_template
)

entity_chain = LLMChain(llm=llm, prompt=entity_prompt)
print("Entity Extraction Chain created successfully!")

Entity Extraction Chain created successfully!


In [13]:
# Test entity extraction with first article
print("Testing Entity Extraction:")
print(f"\nTitle: {test_article['title']}")

entities = entity_chain.run(
    title=test_article['title'],
    content=test_article['content']
)

print(f"\nExtracted Entities:")
print(entities.strip())

Testing Entity Extraction:

Title: Ad sales boost Time Warner profit

Extracted Entities:
TimeWarner, Google, AOL, Warner Bros, United States, Securities and Exchange Commission (SEC), Richard Parsons, Bertelsmann, AOL Europe


In [ ]:
bbc_df['Detected_Topic'] = ''
bbc_df['Summary'] = ''
bbc_df['Key_Entities'] = ''

print("Processing BBC News Articles...")
print(f"Total articles to process: {len(bbc_df)}\n")

# Process each article
for idx, row in bbc_df.iterrows():
    try:
        print(f"Processing article {idx + 1}/{len(bbc_df)}: {row['title'][:50]}...")
        
        # Topic Classification
        topic = topic_chain.run(
            title=row['title'],
            content=row['content']
        ).strip()
        
        # Summarization
        summary = summary_chain.run(
            title=row['title'],
            content=row['content']
        ).strip()
        
        # Entity Extraction
        entities_text = entity_chain.run(
            title=row['title'],
            content=row['content']
        ).strip()
        
        # Parse entities into list
        entities_list = [e.strip() for e in entities_text.split(',') if e.strip()]
        
        # Update dataframe
        bbc_df.at[idx, 'Detected_Topic'] = topic
        bbc_df.at[idx, 'Summary'] = summary
        bbc_df.at[idx, 'Key_Entities'] = entities_list
        
        print(f"  ✓ Topic: {topic}, Entities: {len(entities_list)}")
        
    except Exception as e:
        print(f"  ✗ Error processing article {idx}: {str(e)}")
        bbc_df.at[idx, 'Detected_Topic'] = 'Error'
        bbc_df.at[idx, 'Summary'] = 'Error'
        bbc_df.at[idx, 'Key_Entities'] = []

print("\n✓ All BBC articles processed!")

Processing BBC News Articles...
Total articles to process: 30

Processing article 1/30: Ad sales boost Time Warner profit...
  ✓ Topic: Business, Entities: 14
Processing article 2/30: Dollar gains on Greenspan speech...
  ✓ Topic: Business, Entities: 11
Processing article 3/30: Yukos unit buyer faces loan claim...
  ✓ Topic: Business, Entities: 11
Processing article 4/30: High fuel prices hit BA's profits...
  ✓ Topic: Business, Entities: 9
Processing article 5/30: Pernod takeover talk lifts Domecq...
  ✓ Topic: Business, Entities: 23
Processing article 6/30: Japan narrowly escapes recession...
  ✓ Topic: Business, Entities: 6
Processing article 7/30: Jobs growth still slow in the US...
  ✓ Topic: Business, Entities: 8
Processing article 8/30: India calls for fair trade rules...
  ✓ Topic: Business, Entities: 15
Processing article 9/30: Ethiopia's crop production up 24%...
  ✓ Topic: Business, Entities: 6
Processing article 10/30: Court rejects $280bn tobacco case...
  ✓ Topic: Busines

In [ ]:
print("BBC News Analysis Results:")
print("\nDataFrame Shape:", bbc_df.shape)
print("\nFirst 3 Results:")
for idx in range(min(3, len(bbc_df))):
    row = bbc_df.iloc[idx]
    print(f"\n{'='*80}")
    print(f"Article {idx + 1}: {row['title']}")
    print(f"Original Category: {row['category']}")
    print(f"Detected Topic: {row['Detected_Topic']}")
    print(f"Summary: {row['Summary']}")
    print(f"Key Entities: {row['Key_Entities']}")

# Summary statistics
print(f"\n{'='*80}")
print("\nTopic Distribution (Detected):")
print(bbc_df['Detected_Topic'].value_counts())

print("\nTopic Distribution (Original):")
print(bbc_df['category'].value_counts())

BBC News Analysis Results:

DataFrame Shape: (30, 7)

First 3 Results:

Article 1: Ad sales boost Time Warner profit
Original Category: business
Detected Topic: Business
Summary: Time Warner’s quarterly profit jumped 76% to $1.13 billion for the three months ended December, helped by higher advertising revenue, sales of high‑speed internet and a 2% rise in total sales to $11.1 billion, while the company now holds an 8% stake in Google.  Despite the overall gain, its film division saw a 27% profit decline and AOL lost subscribers, though its underlying profit rose 8% on stronger ad sales; the firm also announced it will restate its 2000‑2003 results and has offered $300 million to settle an SEC investigation.  For the full year, Time Warner posted a 27% increase in profit to $3.36 billion and projected about 5% operating‑earnings growth for 2005.
Key Entities: ['TimeWarner', 'Google', 'AOL', 'Warner\u202fBros', 'US Securities Exchange Commission', 'SEC', 'Richard\u202fParsons', 'Alexand

In [ ]:
bbc_output_path = 'bbc_news_analysis_results.csv'
bbc_df.to_csv(bbc_output_path, index=False)
print(f"✓ BBC analysis results saved to: {bbc_output_path}")


bbc_json_path = 'bbc_news_analysis_results.json'
bbc_df['Key_Entities'] = bbc_df['Key_Entities'].apply(lambda x: x if isinstance(x, list) else [])
bbc_df.to_json(bbc_json_path, orient='records', indent=2)
print(f"✓ BBC analysis results saved to: {bbc_json_path}")

✓ BBC analysis results saved to: bbc_news_analysis_results.csv
✓ BBC analysis results saved to: bbc_news_analysis_results.json


---
# PART 2: Job Postings Analysis

In [ ]:
job_df = pd.read_csv('job_title_des.csv')

print("Dataset Shape:", job_df.shape)
print("\nColumn Names:", job_df.columns.tolist())
print("\nFirst Few Rows:")
print(job_df.head())

print("\nData Types:")
print(job_df.dtypes)

print("\nMissing Values:")
print(job_df.isnull().sum())

Dataset Shape: (2277, 3)

Column Names: ['Unnamed: 0', 'Job Title', 'Job Description']

First Few Rows:
   Unnamed: 0             Job Title  \
0           0     Flutter Developer   
1           1      Django Developer   
2           2      Machine Learning   
3           3         iOS Developer   
4           4  Full Stack Developer   

                                     Job Description  
0  We are looking for hire experts flutter develo...  
1  PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...  
2  Data Scientist (Contractor)\r\n\r\nBangalore, ...  
3  JOB DESCRIPTION:\r\n\r\nStrong framework outsi...  
4  job responsibility full stack engineer – react...  

Data Types:
Unnamed: 0          int64
Job Title          object
Job Description    object
dtype: object

Missing Values:
Unnamed: 0         0
Job Title          0
Job Description    0
dtype: int64


In [ ]:
job_df = job_df.head(25).reset_index(drop=True)
print(f"Limited dataset to {len(job_df)} job postings")
print(f"\nSample Job Titles:")
print(job_df['Job Title'].head(10).tolist())

Limited dataset to 25 job postings

Sample Job Titles:
['Flutter Developer', 'Django Developer', 'Machine Learning', 'iOS Developer', 'Full Stack Developer', 'Java Developer', 'Full Stack Developer', 'JavaScript Developer', 'DevOps Engineer', 'Software Engineer']


## Step 2: Define Job Category Classification Task (10 marks)

Create a LangChain chain to classify jobs into broad categories.

In [ ]:
job_category_template = """You are an expert job categorizer. Analyze the following job posting and classify it into ONE of these domains: Technology/IT, Finance, Marketing, Healthcare, Education, or Others.

Few-shot examples:
1. Job Title: "Python Developer" with description about software development -> Technology/IT
2. Job Title: "Financial Analyst" with description about accounting -> Finance
3. Job Title: "Marketing Manager" with description about campaigns -> Marketing
4. Job Title: "Nurse Practitioner" with description about patient care -> Healthcare
5. Job Title: "High School Teacher" with description about teaching -> Education

Job Title: {job_title}
Job Description: {job_description}

Respond with ONLY the domain name (Technology/IT, Finance, Marketing, Healthcare, Education, or Others). Do not include any explanation."""

job_category_prompt = PromptTemplate(
    input_variables=["job_title", "job_description"],
    template=job_category_template
)

job_category_chain = LLMChain(llm=llm, prompt=job_category_prompt)
print("Job Category Classification Chain created successfully!")

Job Category Classification Chain created successfully!


In [ ]:
skills_template = """Extract all the required technical skills, programming languages, tools, and domain knowledge mentioned in the following job description. List them as a comma-separated string.

Examples of skills: Python, Java, SQL, Project Management, CRM Software, Data Analysis, etc.

Job Title: {job_title}
Job Description: {job_description}

Required Skills (comma-separated):"""

skills_prompt = PromptTemplate(
    input_variables=["job_title", "job_description"],
    template=skills_template
)

skills_chain = LLMChain(llm=llm, prompt=skills_prompt)

# Education Extraction Prompt
education_template = """From the following job description, identify the minimum education level required or preferred (if mentioned). Examples: Bachelor's degree in Computer Science, MBA, High School Diploma, etc.

If no specific education requirement is mentioned, respond with 'Not specified'.

Job Title: {job_title}
Job Description: {job_description}

Education Requirement:"""

education_prompt = PromptTemplate(
    input_variables=["job_title", "job_description"],
    template=education_template
)

education_chain = LLMChain(llm=llm, prompt=education_prompt)

# Experience Extraction Prompt
experience_template = """From the following job description, identify the minimum years of experience or experience level required (if mentioned). Examples: 5+ years, Senior-level, Entry-level, 10 years of experience in management, etc.

If no specific experience requirement is mentioned, respond with 'Not specified'.

Job Title: {job_title}
Job Description: {job_description}

Experience Requirement:"""

experience_prompt = PromptTemplate(
    input_variables=["job_title", "job_description"],
    template=experience_template
)

experience_chain = LLMChain(llm=llm, prompt=experience_prompt)

print("All Job Requirements Extraction Chains created successfully!")

All Job Requirements Extraction Chains created successfully!


In [21]:
# Test with first job posting
test_job = job_df.iloc[0]
print("Testing Job Analysis:")
print(f"\nJob Title: {test_job['Job Title']}")
print(f"\nDescription (first 300 chars):")
print(test_job['Job Description'][:300] + "...")

# Test category classification
job_category = job_category_chain.run(
    job_title=test_job['Job Title'],
    job_description=test_job['Job Description']
).strip()
print(f"\nDetected Category: {job_category}")

# Test skills extraction
skills = skills_chain.run(
    job_title=test_job['Job Title'],
    job_description=test_job['Job Description']
).strip()
print(f"\nRequired Skills: {skills}")

# Test education extraction
education = education_chain.run(
    job_title=test_job['Job Title'],
    job_description=test_job['Job Description']
).strip()
print(f"\nEducation Requirement: {education}")

# Test experience extraction
experience = experience_chain.run(
    job_title=test_job['Job Title'],
    job_description=test_job['Job Description']
).strip()
print(f"\nExperience Requirement: {experience}")

Testing Job Analysis:

Job Title: Flutter Developer

Description (first 300 chars):
We are looking for hire experts flutter developer. So you are eligible this post then apply your resume.
Job Types: Full-time, Part-time
Salary: ₹20,000.00 - ₹40,000.00 per month
Benefits:
Flexible schedule
Food allowance
Schedule:
Day shift
Supplemental Pay:
Joining bonus
Overtime pay
Ex...

Detected Category: Technology/IT

Required Skills: Flutter, Software Development

Education Requirement: Not specified

Experience Requirement: 1 year (Preferred)


In [22]:
# Initialize result columns
job_df['Predicted_Category'] = ''
job_df['Required_Skills'] = ''
job_df['Education_Required'] = ''
job_df['Experience_Required'] = ''

print("Processing Job Postings...")
print(f"Total postings to process: {len(job_df)}\n")

# Process each job posting
for idx, row in job_df.iterrows():
    try:
        print(f"Processing job {idx + 1}/{len(job_df)}: {row['Job Title'][:40]}...")
        
        # Job Category Classification
        category = job_category_chain.run(
            job_title=row['Job Title'],
            job_description=row['Job Description']
        ).strip()
        
        # Skills Extraction
        skills = skills_chain.run(
            job_title=row['Job Title'],
            job_description=row['Job Description']
        ).strip()
        
        # Education Extraction
        education = education_chain.run(
            job_title=row['Job Title'],
            job_description=row['Job Description']
        ).strip()
        
        # Experience Extraction
        experience = experience_chain.run(
            job_title=row['Job Title'],
            job_description=row['Job Description']
        ).strip()
        
        # Parse skills into list
        skills_list = [s.strip() for s in skills.split(',') if s.strip()]
        
        # Update dataframe
        job_df.at[idx, 'Predicted_Category'] = category
        job_df.at[idx, 'Required_Skills'] = skills_list
        job_df.at[idx, 'Education_Required'] = education
        job_df.at[idx, 'Experience_Required'] = experience
        
        print(f"  ✓ Category: {category}, Skills: {len(skills_list)}")
        
    except Exception as e:
        print(f"  ✗ Error processing job {idx}: {str(e)}")
        job_df.at[idx, 'Predicted_Category'] = 'Error'
        job_df.at[idx, 'Required_Skills'] = []
        job_df.at[idx, 'Education_Required'] = 'Error'
        job_df.at[idx, 'Experience_Required'] = 'Error'

print("\n✓ All job postings processed!")

Processing Job Postings...
Total postings to process: 25

Processing job 1/25: Flutter Developer...
  ✓ Category: Technology/IT, Skills: 2
Processing job 2/25: Django Developer...
  ✓ Category: Technology/IT, Skills: 10
Processing job 3/25: Machine Learning...
  ✓ Category: Technology/IT, Skills: 15
Processing job 4/25: iOS Developer...
  ✓ Category: Technology/IT, Skills: 20
Processing job 5/25: Full Stack Developer...
  ✓ Category: Technology/IT, Skills: 28
Processing job 6/25: Java Developer...
  ✓ Category: Technology/IT, Skills: 20
Processing job 7/25: Full Stack Developer...
  ✓ Category: Technology/IT, Skills: 20
Processing job 8/25: JavaScript Developer...
  ✓ Category: Technology/IT, Skills: 8
Processing job 9/25: DevOps Engineer...
  ✓ Category: Technology/IT, Skills: 37
Processing job 10/25: Software Engineer...
  ✓ Category: Technology/IT, Skills: 16
Processing job 11/25: Database Administrator...
  ✓ Category: Technology/IT, Skills: 28
Processing job 12/25: Machine Learnin

In [ ]:

print("Job Postings Analysis Results:")
print("\nDataFrame Shape:", job_df.shape)
print("\nFirst 3 Results:")
for idx in range(min(3, len(job_df))):
    row = job_df.iloc[idx]
    print(f"\n{'='*80}")
    print(f"Job {idx + 1}: {row['Job Title']}")
    print(f"Predicted Category: {row['Predicted_Category']}")
    print(f"Required Skills: {row['Required_Skills']}")
    print(f"Education Required: {row['Education_Required']}")
    print(f"Experience Required: {row['Experience_Required']}")

# Summary statistics
print(f"\n{'='*80}")
print("\nCategory Distribution:")
print(job_df['Predicted_Category'].value_counts())

Job Postings Analysis Results:

DataFrame Shape: (25, 7)

First 3 Results:

Job 1: Flutter Developer
Predicted Category: Technology/IT
Required Skills: ['Flutter', 'Software Development']
Education Required: Not specified
Experience Required: 1 year (Preferred)

Job 2: Django Developer
Predicted Category: Technology/IT
Required Skills: ['Python', 'Django', 'Flask', 'REST', 'RPC', 'Linux', 'SQL', 'JSON', 'PyUnit', 'Automated unit testing']
Education Required: Not specified
Experience Required: Not specified

Job 3: Machine Learning
Predicted Category: Technology/IT
Required Skills: ['Python', 'Java', 'Statistics', 'Applied Mathematics', 'Machine Learning', 'Deep Learning', 'Big Data', 'Spark', 'PyTorch', 'TensorFlow', 'Keras', 'Data Engineering', 'Software Development', 'Telecommunication', 'Fraud Prevention']
Education Required: Bachelor’s degree (any graduate) in Computer Science, Mathematics or a related field (M.Sc. in the same areas is also mentioned as preferred).
Experience Requi

In [24]:
# Save job results to CSV
job_output_path = 'job_analysis_results.csv'
job_df.to_csv(job_output_path, index=False)
print(f"✓ Job analysis results saved to: {job_output_path}")

# Also save as JSON for better readability
job_json_path = 'job_analysis_results.json'
job_df['Required_Skills'] = job_df['Required_Skills'].apply(lambda x: x if isinstance(x, list) else [])
job_df.to_json(job_json_path, orient='records', indent=2)
print(f"✓ Job analysis results saved to: {job_json_path}")

✓ Job analysis results saved to: job_analysis_results.csv
✓ Job analysis results saved to: job_analysis_results.json


### Output Files:
1. `bbc_news_analysis_results.csv` - News article analysis results
2. `bbc_news_analysis_results.json` - News article analysis (JSON format)
3. `job_analysis_results.csv` - Job posting analysis results
4. `job_analysis_results.json` - Job posting analysis (JSON format)


In [ ]:
def process_full_bbc_dataset():
    """Process entire BBC news dataset for bonus marks"""
    bbc_full = pd.read_csv('bbc-news-data.csv', sep='\t')
    print(f"Processing {len(bbc_full)} articles...")
    
    bbc_full['Detected_Topic'] = ''
    bbc_full['Summary'] = ''
    bbc_full['Key_Entities'] = ''
    
    for idx, row in bbc_full.iterrows():
        if idx % 50 == 0:
            print(f"Progress: {idx}/{len(bbc_full)}")
        
        try:
            topic = topic_chain.run(title=row['title'], content=row['content']).strip()
            summary = summary_chain.run(title=row['title'], content=row['content']).strip()
            entities_text = entity_chain.run(title=row['title'], content=row['content']).strip()
            entities_list = [e.strip() for e in entities_text.split(',') if e.strip()]
            
            bbc_full.at[idx, 'Detected_Topic'] = topic
            bbc_full.at[idx, 'Summary'] = summary
            bbc_full.at[idx, 'Key_Entities'] = entities_list
        except:
            pass
    
    bbc_full.to_csv('bbc_full_analysis.csv', index=False)
    print("✓ Full BBC dataset processing complete!")
    return bbc_full

bbc_full_results = process_full_bbc_dataset()

Bonus processing functions defined (commented out to avoid rate limits)
Uncomment the functions above to process full datasets for bonus marks.


In [ ]:
bbc_full_results